In [82]:
import requests
import re

# Map single-letter to three-letter amino acid codes
aa_1to3 = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp',
    'C': 'Cys', 'Q': 'Gln', 'E': 'Glu', 'G': 'Gly',
    'H': 'His', 'I': 'Ile', 'L': 'Leu', 'K': 'Lys',
    'M': 'Met', 'F': 'Phe', 'P': 'Pro', 'S': 'Ser',
    'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val'
}

def parse_mutation(mutation):
    # Accepts mutation in the form "V50M"

    match = re.match(r"([A-Za-z]{1,3})\s*(\d+)\s*([A-Za-z]{1,3})", mutation.replace('.', '').replace('p', ''))
    if not match:
        raise ValueError("Mutation format not recognized")
    from_aa, pos, to_aa = match.groups()
    pos = int(pos)
    # Normalize to single-letter codes
    from_aa_1 = from_aa if len(from_aa) == 1 else [k for k, v in aa_1to3.items() if v.lower() == from_aa.lower()][0]
    to_aa_1 = to_aa if len(to_aa) == 1 else [k for k, v in aa_1to3.items() if v.lower() == to_aa.lower()][0]
    # Three-letter codes
    from_aa_3 = aa_1to3[from_aa_1]
    to_aa_3 = aa_1to3[to_aa_1]
    # Both original and +20 offset
    positions = [pos, pos+20]
    variants = []
    for p in positions:
        variants.extend([
            f"{from_aa_1}{p}{to_aa_1}",
            f"{from_aa_3}{p}{to_aa_3}",
            #f"p.{from_aa_1}{p}{to_aa_1}",
            #f"p.{from_aa_3}{p}{to_aa_3}",
        ])
    return variants

"""
LIGAND_CODE_MAP = {
    "tafamidis": "3MI",
    "acoramidis": "16V",
    "tolcapone": "TCW",
    "diflunisal": "1FL"
}
"""

def search_transthyretin_pdb(mutation="V50M", ligand=None):
    ligand_variants = ["tafamidis", "acoramidis", "tolcapone", "diflunisal"]
    if ligand:
        ligand = ligand.lower()
        ligands_to_search = [ligand]
    else:
        ligands_to_search = ligand_variants

    if mutation != "WildType":
      mutation_variants = parse_mutation(mutation)
    else:
      mutation_variants = ["WildType", "wildtype", "wild-type"]
    found_ids = []

    for mut in mutation_variants:
        for lig in ligands_to_search:
            #lig_code = LIGAND_CODE_MAP.get(lig.lower())
            #if not lig_code:
            #    continue  # Skip unknown ligands
            #print(f"Searching for {mut} and {lig_code}")
            query = {
                "query": {
                    "type": "group",
                    "logical_operator": "and",
                    "nodes": [
                        {
                            "type": "terminal",
                            "service": "text",
                            "parameters": {
                                "attribute": "struct.title",
                                "operator": "contains_phrase",
                                "value": "transthyretin"
                            }
                        },
                        {
                            "type": "terminal",
                            "service": "text",
                            "parameters": {
                                "attribute": "struct.title",
                                "operator": "contains_words",
                                "value": mut
                            }
                        },
                        {
                            "type": "terminal",
                            "service": "text",
                            "parameters": {
                                "attribute": "struct.title",
                                "operator": "contains_words",
                                "value": lig
                            }
                        }
                    ]
                },
                "return_type": "entry"
            }
            try:
                response = requests.post(
                    "https://search.rcsb.org/rcsbsearch/v2/query?json",
                    json=query,
                    timeout=10
                )
                if response.ok:
                    results = response.json()
                    if "result_set" in results and results["result_set"]:
                        found_ids.extend([r["identifier"] for r in results["result_set"]])
            except Exception as e:
              continue
                #print(f"Request failed: {e}")
    return found_ids[0] if found_ids else None

In [73]:
# Example usage:
pdb_id = search_transthyretin_pdb(mutation="A97S", ligand="tafamidis")
print(pdb_id)

8YQD


In [83]:
mutations = ['WildType', 'A109S', 'A109V', 'A120S', 'A19D', 'A25S', 'A25T', 'A36D', 'A36P',
       'A45D', 'A45G', 'A45T', 'A81T', 'A81V', 'A91S', 'A97G', 'A97S',
       'C10R', 'D18E', 'D18G', 'D18N', 'D38A', 'D38V', 'D39V', 'D74H',
       'E42D', 'E42G', 'E51G', 'E54D', 'E54G', 'E54K', 'E54L', 'E54Q',
       'E61G', 'E61K', 'E62K', 'E72G', 'E89K', 'E89Q', 'E92K', 'F33C',
       'F33I', 'F33L', 'F33V', 'F44L', 'F44S', 'F44Y', 'F64I', 'F64L',
       'F64S', 'G101S', 'G47A', 'G47E', 'G47R', 'G47V', 'G53A', 'G53E',
       'G53R', 'G57R', 'G67E', 'G67R', 'G83R', 'H56R', 'H88R', 'H90D',
       'H90N', 'I107F', 'I107M', 'I107V', 'I68L', 'I73V', 'I84N', 'I84S',
       'I84T', 'K35N', 'K35T', 'K70N', 'L111M', 'L12P', 'L55P', 'L55Q',
       'L55R', 'L58H', 'L58R', 'M13I', 'P102R', 'P113T', 'P125S', 'P24S',
       'R103S', 'R104C', 'R104H', 'R21Q', 'R34G', 'R34T', 'S112I', 'S23N',
       'S50I', 'S50R', 'S52P', 'S77F', 'S77Y', 'T119M', 'T49A', 'T49I',
       'T49P', 'T49S', 'T59K', 'T59R', 'T60A', 'V122A', 'V122I', 'V20I',
       'V28M', 'V28S', 'V30A', 'V30G', 'V30L', 'V30M', 'V32A', 'V32G',
       'V71A', 'V93M', 'V94A', 'W41L', 'Y114C', 'Y114H',
       'Y114S', 'Y116S', 'Y69H', 'Y69I', 'Y78F']
ligands = ["tafamidis", "acoramidis", "diflunisal", "tolcapone"]
len(mutations)

132

In [84]:
import pandas as pd

rows = []
for i, mutation in enumerate(mutations):
  print(f"\r{i+1}/{len(mutations)}", end="")
  row = [mutation]
  for ligand in ligands:
    pdb_id = search_transthyretin_pdb(mutation=mutation, ligand=ligand)
    row.append(pdb_id)
  rows.append(row)

df = pd.DataFrame(rows, columns=["mutation"] + ligands)
df.to_csv("experimental_pdbs_complex.csv", index=False)

132/132